# 4. SCF Convergence Time-Series Analysis

Treat DFT self-consistent-field iterations as a time series to:
1. **Detect charge sloshing** via FFT spectral analysis
2. **Predict convergence step** from early iterations
3. **Recommend VASP algorithm** (Damped/Fast/All) from convergence signature
4. **Track ionic relaxation** quality across geometry steps

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 4.1 Simulate SCF trajectories
We simulate three types of convergence behaviour.

In [ ]:
rng = np.random.default_rng(42)

# 1. Healthy exponential decay (well-behaved insulator)
n = 50
t = np.arange(n)
healthy = 0.5 * np.exp(-0.25 * t) + rng.normal(0, 0.0005, n)
healthy = np.abs(healthy)

# 2. Charge sloshing (metallic surface with poor mixing)
sloshing = 0.3 * np.sin(0.35 * np.pi * t) * np.exp(-0.01 * t) + 0.02
sloshing = np.abs(sloshing)

# 3. Slow convergence (near-gap semiconductor)
slow = 0.8 * np.exp(-0.04 * t) + rng.normal(0, 0.001, n)
slow = np.abs(slow)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, dE, title, color in zip(axes,
    [healthy, sloshing, slow],
    ['Healthy (exponential)', 'Charge sloshing', 'Slow convergence'],
    ['green', 'red', 'orange']):
    ax.semilogy(dE, 'o-', color=color, markersize=3, lw=1)
    ax.axhline(1e-5, color='gray', ls='--', label='EDIFF=1e-5')
    ax.set_xlabel('SCF step'); ax.set_ylabel('|ΔE| (eV)')
    ax.set_title(title); ax.legend(fontsize=8)
plt.suptitle('Three types of SCF convergence', fontweight='bold')
plt.tight_layout()
plt.show()

## 4.2 Charge Sloshing Detection (FFT)
The detector computes the FFT of log|ΔE| and looks for dominant AC components.

In [ ]:
from science.time_series.scf_convergence import (
    SCFTrajectory, ChargeSloshingDetector, ConvergenceRatePredictor,
    AlgorithmRecommender, analyse_scf, IonicConvergenceTracker,
)

detector = ChargeSloshingDetector()

for name, dE in [('Healthy', healthy), ('Sloshing', sloshing), ('Slow', slow)]:
    traj = SCFTrajectory(dE=dE.tolist(), nelm=100, ediff=1e-5)
    result = detector.detect(traj)
    print(f'{name:12s}  sloshing={result.is_sloshing!s:5s}  '
          f'freq={result.dominant_frequency:.3f}  '
          f'amp={result.amplitude:.3f}  '
          f'decay={result.decay_rate:+.3f}  '
          f'confidence={result.confidence:.2f}')
    print(f'             Remedy: {result.remedy[:80]}...\n')

## 4.3 Convergence Rate Prediction
Fit log|ΔE| = log(A) - λn and predict the step at which EDIFF is reached.

In [ ]:
predictor = ConvergenceRatePredictor(window=15)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, dE, name in zip(axes,
    [healthy, sloshing, slow],
    ['Healthy', 'Sloshing', 'Slow']):
    traj = SCFTrajectory(dE=dE.tolist(), nelm=200, ediff=1e-5)
    pred = predictor.predict(traj)

    ax.semilogy(dE, 'o-', markersize=3, lw=1)
    ax.axhline(1e-5, color='gray', ls='--', lw=0.8)

    # Show prediction line
    if pred.convergence_rate > 0:
        fit_x = np.arange(min(len(dE), pred.predicted_step + 5))
        A = np.exp(np.log(dE[0] if dE[0] > 0 else 1e-10))
        fit_y = A * np.exp(-pred.convergence_rate * fit_x)
        ax.semilogy(fit_x, fit_y, 'r--', lw=1.5, alpha=0.7, label='exponential fit')
        ax.axvline(pred.predicted_step, color='green', ls=':', label=f'predicted: step {pred.predicted_step}')

    ax.set_title(f'{name}\nλ={pred.convergence_rate:.3f}, R²={pred.r_squared:.3f}\n'
                 f'n_conv={pred.predicted_step}, {pred.confidence} conf')
    ax.set_xlabel('SCF step'); ax.set_ylabel('|ΔE| (eV)')
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

## 4.4 Algorithm Recommender
Maps convergence signature → optimal VASP ALGO/AMIX settings.

In [ ]:
recommender = AlgorithmRecommender()

scenarios = [
    ('Metal, fast convergence', healthy, True, False),
    ('Metal, sloshing', sloshing, True, False),
    ('Oxide with d-electrons', slow, False, True),
]

for name, dE, is_metal, has_d in scenarios:
    traj = SCFTrajectory(dE=dE.tolist(), nelm=200, ediff=1e-5)
    slosh = detector.detect(traj)
    pred = predictor.predict(traj)
    rec = recommender.recommend(slosh, pred, is_metal=is_metal, has_d_electrons=has_d)
    print(f'--- {name} ---')
    print(f'  Recommended: {rec.algo}')
    print(f'  Settings:    {rec.settings}')
    print(f'  Rationale:   {rec.rationale[:100]}...')
    print()

## 4.5 Full Analysis Pipeline (one-liner)

In [ ]:
traj = SCFTrajectory(dE=sloshing.tolist(), nelm=100, ediff=1e-5)
report = analyse_scf(traj, is_metal=True, has_d_electrons=False)
print(report)

## 4.6 Ionic Convergence Tracker
Track SCF quality across geometry relaxation steps.

In [ ]:
tracker = IonicConvergenceTracker()

# Simulate 10 ionic steps with varying SCF difficulty
for i in range(10):
    n_scf = 15 + int(10 * np.sin(i * 0.5)) + rng.integers(0, 5)
    dE_ionic = (0.5 * np.exp(-0.2 * np.arange(n_scf))).tolist()
    tracker.add_ionic_step(SCFTrajectory(dE=dE_ionic, nelm=60, ediff=1e-5))

print(tracker.report())

# Plot SCF steps per ionic step
counts = tracker.scf_step_counts()
quality = tracker.convergence_quality_series()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
ax1.bar(range(len(counts)), counts, color='steelblue')
ax1.set_ylabel('SCF steps')
ax1.set_title('SCF cost per ionic step')
ax2.plot(quality, 'o-', color='green')
ax2.set_ylabel('Convergence quality')
ax2.set_xlabel('Ionic step')
ax2.set_ylim(-0.1, 1.1)
plt.tight_layout()
plt.show()